# Alexander gene sets — Enrichr enrichment

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from maayanlab_bioinformatics.api import EnrichrUserList

GENE_SETS = Path("gene_sets")
MUTATIONS = ["R239H", "R239C", "R79C", "R79H", "R239G"]
DIRECTIONS = ["up", "down"]
LIBRARIES = [
    "GO_Biological_Process_2023",
    "KEGG_2021_Human",
    "Reactome_Pathways_2024",
    "MSigDB_Hallmark_2020",
]
TOP_N = 10

In [2]:
def load(mut, direction):
    p = GENE_SETS / f"{mut}_{direction}.txt"
    return [l.strip() for l in p.read_text().splitlines() if l.strip()]

gene_sets = {(m, d): load(m, d) for m in MUTATIONS for d in DIRECTIONS}
pd.Series({f"{m}_{d}": len(g) for (m, d), g in gene_sets.items()}, name="n_genes")

R239H_up       61
R239H_down    123
R239C_up       33
R239C_down     67
R79C_up        63
R79C_down      69
R79H_up        57
R79H_down      75
R239G_up       54
R239G_down     89
Name: n_genes, dtype: int64

In [3]:
frames, links = [], {}
for (m, d), genes in gene_sets.items():
    if not genes:
        print(f"{m}_{d}: empty gene set, skipped")
        continue
    ul = EnrichrUserList(genes, f"{m}_{d}")
    links[f"{m}_{d}"] = ul.link
    n = 0
    for lib in LIBRARIES:
        df = ul[lib]
        df.insert(0, "library", lib)
        df.insert(0, "direction", d)
        df.insert(0, "mutation", m)
        frames.append(df)
        n += len(df)
    print(f"{m}_{d}: {n} enriched terms across {len(LIBRARIES)} libraries")

enr = pd.concat(frames, ignore_index=True)
enr["n_overlap"] = enr["overlapping_genes"].str.len()
enr.shape

R239H_up: 1350 enriched terms across 4 libraries


R239H_down: 1836 enriched terms across 4 libraries


R239C_up: 709 enriched terms across 4 libraries


R239C_down: 1075 enriched terms across 4 libraries


R79C_up: 1008 enriched terms across 4 libraries


...

R79H_up: 1061 enriched terms across 4 libraries


R79H_down: 1228 enriched terms across 4 libraries


R239G_up: 1035 enriched terms across 4 libraries


R239G_down: 1351 enriched terms across 4 libraries


(11783, 11)

## Shareable Enrichr links (one per gene set)

In [ ]:
pd.Series(links, name="enrichr_link")

,enrichr_link
R239H_up,https://maayanlab.cloud/Enrichr/enrich?dataset...
R239H_down,https://maayanlab.cloud/Enrichr/enrich?dataset...
R239C_up,https://maayanlab.cloud/Enrichr/enrich?dataset...
R239C_down,https://maayanlab.cloud/Enrichr/enrich?dataset...
R79C_up,https://maayanlab.cloud/Enrichr/enrich?dataset...
R79C_down,https://maayanlab.cloud/Enrichr/enrich?dataset...
R79H_up,https://maayanlab.cloud/Enrichr/enrich?dataset...
R79H_down,https://maayanlab.cloud/Enrichr/enrich?dataset...
R239G_up,https://maayanlab.cloud/Enrichr/enrich?dataset...
R239G_down,https://maayanlab.cloud/Enrichr/enrich?dataset...


In [5]:
def top_terms(mut, direction, n=TOP_N):
    sub = enr[(enr.mutation == mut) & (enr.direction == direction)]
    cols = ["library", "term", "pvalue", "adjusted_pvalue", "combinedscore", "n_overlap", "overlapping_genes"]
    return sub.sort_values("adjusted_pvalue").head(n)[cols].reset_index(drop=True)

pd.set_option("display.max_colwidth", 70)

## Top enriched terms per gene set

In [6]:
for d in DIRECTIONS:
    print(f"\n========== R239H {d} ==========")
    display(top_terms("R239H", d))


========== R239H up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,KEGG_2021_Human,Glutathione metabolism,0.000027,0.004248,276.591660,4,"[GSTM4, GSTM1, ODC1, GPX7]"
1,KEGG_2021_Human,Hepatocellular carcinoma,0.000160,0.012423,94.660925,5,"[GSTM4, GSTM1, WNT5A, HMOX1, SOS1]"
2,KEGG_2021_Human,Fluid shear stress and atherosclerosis,0.000858,0.043732,72.691504,4,"[GSTM4, PIAS4, GSTM1, HMOX1]"
3,KEGG_2021_Human,Pathways in cancer,0.001129,0.043732,32.596835,7,"[GSTM4, NCOA1, GSTM1, WNT5A, ADCY3, HMOX1, SOS1]"
4,MSigDB_Hallmark_2020,Adipogenesis,0.003246,0.084391,40.506625,4,"[C3, PPP1R15B, MIGA2, RAB34]"
5,Reactome_Pathways_2024,Heme Signaling,0.000398,0.084825,183.112593,3,"[NCOA1, HMOX1, APOA1]"
6,Reactome_Pathways_2024,Metabolism of Lipids,0.000456,0.084825,34.112133,9,"[GSTM4, NCOA1, PIAS4, MIGA2, ARSL, APOA1, ARSD, HMGCLL1, B4GALT5]"
7,Reactome_Pathways_2024,The Activation of Arylsulfatases,0.000698,0.084825,446.255637,2,"[ARSL, ARSD]"
8,Reactome_Pathways_2024,Glycosphingolipid Metabolism,0.000739,0.084825,134.823303,3,"[ARSL, ARSD, B4GALT5]"
9,Reactome_Pathways_2024,Post-translational Protein Modification,0.001314,0.120612,20.795784,12,"[NCOA1, PIAS4, C3, ITIH2, RAB42, ARSL, RAB34, LYPD1, APOA1, KDELR2..."



========== R239H down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,MSigDB_Hallmark_2020,Hypoxia,0.000004,0.000148,100.907314,9,"[SLC6A6, ERRFI1, HOXB9, CSRP2, STC2, GAA, PLAUR, SLC2A3, PYGM]"
1,GO_Biological_Process_2023,Glucan Catabolic Process (GO:0009251),0.000004,0.001981,2039.765691,3,"[PYGB, GAA, PYGM]"
2,GO_Biological_Process_2023,Glycogen Catabolic Process (GO:0005980),0.000004,0.001981,2039.765691,3,"[PYGB, GAA, PYGM]"
3,GO_Biological_Process_2023,Glycogen Metabolic Process (GO:0005977),0.000005,0.001981,544.037697,4,"[PYGB, GAA, PRKAG2, PYGM]"
4,MSigDB_Hallmark_2020,Interferon Alpha Response,0.000333,0.005988,72.976710,5,"[BST2, PROCR, ELF1, DHX58, TXNIP]"
5,MSigDB_Hallmark_2020,Estrogen Response Early,0.001496,0.010772,33.845178,6,"[SOX3, ELF1, KRT18, STC2, KRT8, FLNB]"
6,MSigDB_Hallmark_2020,Estrogen Response Late,0.001496,0.010772,33.845178,6,"[SOX3, PRKAR2B, AGR2, FLNB, CXCL14, ASS1]"
7,MSigDB_Hallmark_2020,Epithelial Mesenchymal Transition,0.001496,0.010772,33.845178,6,"[NNMT, LRP1, CALD1, PLAUR, DKK1, SNTB1]"
8,GO_Biological_Process_2023,Actin Filament Bundle Assembly (GO:0051017),0.000049,0.011739,228.511934,4,"[BAIAP2L1, CALD1, LCP1, AIF1L]"
9,GO_Biological_Process_2023,Actin Filament Bundle Organization (GO:0061572),0.000049,0.011739,228.511934,4,"[BAIAP2L1, CALD1, LCP1, AIF1L]"


In [7]:
for d in DIRECTIONS:
    print(f"\n========== R239C {d} ==========")
    display(top_terms("R239C", d))


========== R239C up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,GO_Biological_Process_2023,Transcription By RNA Polymerase II (GO:0006366),0.003845,0.129512,58.194417,3,"[NCOA1, CCNH, TAF5L]"
1,GO_Biological_Process_2023,Negative Regulation Of Receptor Binding (GO:1900121),0.014755,0.129512,328.712393,1,[HFE]
2,GO_Biological_Process_2023,Negative Regulation Of Toll-Like Receptor 4 Signaling Pathway (GO:...,0.014755,0.129512,328.712393,1,[DAB2IP]
3,GO_Biological_Process_2023,Negative Regulation Of Vascular Endothelial Growth Factor Signalin...,0.014755,0.129512,328.712393,1,[DAB2IP]
4,GO_Biological_Process_2023,Ornithine Metabolic Process (GO:0006591),0.014755,0.129512,328.712393,1,[ODC1]
5,GO_Biological_Process_2023,Regulation Of Receptor Binding (GO:1900120),0.014755,0.129512,328.712393,1,[HFE]
6,GO_Biological_Process_2023,Pinocytosis (GO:0006907),0.014755,0.129512,328.712393,1,[RAB34]
7,GO_Biological_Process_2023,"Regulation Of CD8-positive, Alpha-Beta T Cell Activation (GO:2001185)",0.014755,0.129512,328.712393,1,[HFE]
8,GO_Biological_Process_2023,Epithelial Cell Apoptotic Process (GO:1904019),0.016382,0.129512,284.928150,1,[DAB2IP]
9,GO_Biological_Process_2023,Regulation Of Chromatin Binding (GO:0035561),0.016382,0.129512,284.928150,1,[ZNF618]



========== R239C down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,GO_Biological_Process_2023,Regulation Of Protein Targeting To Membrane (GO:0090313),0.000002,0.001492,685.737496,4,"[CEMIP, ERBB2, INPP5K, MIEF2]"
1,GO_Biological_Process_2023,Positive Regulation Of Protein Targeting To Membrane (GO:0090314),0.000061,0.020410,453.191899,3,"[CEMIP, ERBB2, MIEF2]"
2,GO_Biological_Process_2023,Ruffle Assembly (GO:0097178),0.000110,0.024070,1863.514013,2,"[INPP5K, AIF1L]"
3,GO_Biological_Process_2023,Positive Regulation Of Protein Localization To Membrane (GO:1905477),0.000143,0.024070,148.815176,4,"[CEMIP, LRP1, ERBB2, MIEF2]"
4,GO_Biological_Process_2023,Neurogenesis (GO:0022008),0.000593,0.079808,150.580016,3,"[BTBD1, ERBB2, LRP4]"
5,GO_Biological_Process_2023,Cellular Response To Exogenous dsRNA (GO:0071360),0.000980,0.091633,353.865351,2,"[STING1, RIGI]"
6,GO_Biological_Process_2023,Positive Regulation Of Potassium Ion Transport (GO:0043268),0.001129,0.091633,319.983578,2,"[NOS1AP, ANK2]"
7,GO_Biological_Process_2023,Ruffle Organization (GO:0031529),0.001287,0.091633,291.361590,2,"[INPP5K, AIF1L]"
8,GO_Biological_Process_2023,Positive Regulation Of Potassium Ion Transmembrane Transport (GO:1...,0.001455,0.091633,266.898294,2,"[NOS1AP, ANK2]"
9,GO_Biological_Process_2023,Cellular Response To dsRNA (GO:0071359),0.001455,0.091633,266.898294,2,"[STING1, RIGI]"


In [8]:
for d in DIRECTIONS:
    print(f"\n========== R79C {d} ==========")
    display(top_terms("R79C", d))


========== R79C up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,Reactome_Pathways_2024,EPHA-mediated Growth Cone Collapse,0.000103,0.031618,351.663731,3,"[EPHA5, EPHA7, MYH10]"
1,Reactome_Pathways_2024,Glycosphingolipid Metabolism,0.000812,0.083396,128.611977,3,"[ARSL, ARSD, B4GALT5]"
2,Reactome_Pathways_2024,The Activation of Arylsulfatases,0.000745,0.083396,427.761017,2,"[ARSL, ARSD]"
3,KEGG_2021_Human,Adrenergic signaling in cardiomyocytes,0.001284,0.148345,61.183839,4,"[ADCY3, CALML3, PPP2R3A, SLC8A1]"
4,KEGG_2021_Human,Vascular smooth muscle contraction,0.008537,0.148345,36.287158,3,"[ADCY3, CALML3, MYH10]"
5,KEGG_2021_Human,Basal transcription factors,0.008861,0.148345,71.689549,2,"[CCNH, TAF5L]"
6,KEGG_2021_Human,Estrogen signaling pathway,0.009255,0.148345,34.600617,3,"[NCOA1, ADCY3, CALML3]"
7,KEGG_2021_Human,Apelin signaling pathway,0.009255,0.148345,34.600617,3,"[ADCY3, CALML3, SLC8A1]"
8,KEGG_2021_Human,"Valine, leucine and isoleucine degradation",0.010035,0.148345,65.239659,2,"[ACAA2, HMGCLL1]"
9,KEGG_2021_Human,Vibrio cholerae infection,0.010855,0.148345,61.449068,2,"[ADCY3, KDELR2]"



========== R79C down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,MSigDB_Hallmark_2020,IL-2/STAT5 Signaling,0.000622,0.009863,58.677780,5,"[NRP1, ECM1, SPP1, DENND5A, SLC2A3]"
1,MSigDB_Hallmark_2020,KRAS Signaling Up,0.000636,0.009863,58.194679,5,"[ALDH1A3, NRP1, HKDC1, SPP1, LCP1]"
2,KEGG_2021_Human,Starch and sucrose metabolism,0.000259,0.026895,226.393030,3,"[PYGB, HKDC1, PYGM]"
3,MSigDB_Hallmark_2020,Estrogen Response Early,0.005055,0.035398,32.762128,4,"[SOX3, ELF1, STC2, AFF1]"
4,MSigDB_Hallmark_2020,Epithelial Mesenchymal Transition,0.005055,0.035398,32.762128,4,"[ECM1, SPP1, ITGA5, LOXL2]"
5,MSigDB_Hallmark_2020,Hedgehog Signaling,0.006851,0.035398,87.052836,2,"[NRP1, NRP2]"
6,MSigDB_Hallmark_2020,Angiogenesis,0.006851,0.035398,87.052836,2,"[NRP1, SPP1]"
7,MSigDB_Hallmark_2020,Reactive Oxygen Species Pathway,0.012426,0.055030,55.414403,2,"[SBNO2, FTL]"
8,Reactome_Pathways_2024,Post-translational Protein Phosphorylation,0.000512,0.055911,89.753466,4,"[IGFBP5, STC2, SPP1, MELTF]"
9,Reactome_Pathways_2024,L1CAM Interactions,0.000789,0.055911,75.109147,4,"[NRP1, GAP43, NRP2, ITGA5]"


In [9]:
for d in DIRECTIONS:
    print(f"\n========== R79H {d} ==========")
    display(top_terms("R79H", d))


========== R79H up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,KEGG_2021_Human,Glutathione metabolism,0.000021,0.003369,305.173984,4,"[GSTM1, ODC1, GPX7, CHAC2]"
1,KEGG_2021_Human,Prolactin signaling pathway,0.001049,0.084467,113.053360,3,"[SOCS2, SOS1, MAPK12]"
2,KEGG_2021_Human,GnRH signaling pathway,0.002375,0.116296,74.053830,3,"[CALML3, SOS1, MAPK12]"
3,KEGG_2021_Human,Human cytomegalovirus infection,0.003865,0.116296,37.419291,4,"[HLA-B, CALML3, SOS1, MAPK12]"
4,KEGG_2021_Human,"Growth hormone synthesis, secretion and action",0.004757,0.116296,50.784930,3,"[SOCS2, SOS1, MAPK12]"
5,KEGG_2021_Human,Neurotrophin signaling pathway,0.004757,0.116296,50.784930,3,"[CALML3, SOS1, MAPK12]"
6,KEGG_2021_Human,Dopaminergic synapse,0.006341,0.116296,43.183665,3,"[CALML3, PPP2R3A, MAPK12]"
7,KEGG_2021_Human,Estrogen signaling pathway,0.007025,0.116296,40.721072,3,"[NCOA1, CALML3, SOS1]"
8,KEGG_2021_Human,Insulin signaling pathway,0.007025,0.116296,40.721072,3,"[SOCS2, CALML3, SOS1]"
9,KEGG_2021_Human,Fluid shear stress and atherosclerosis,0.007310,0.116296,39.796168,3,"[GSTM1, CALML3, MAPK12]"



========== R79H down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,GO_Biological_Process_2023,Endosomal Transport (GO:0016197),0.000059,0.048664,96.145328,6,"[TINAGL1, ALS2, VPS50, ALMS1, DENND5A, RILP]"
1,GO_Biological_Process_2023,Positive Regulation Of Cholesterol Storage (GO:0010886),0.000138,0.052691,1617.435782,2,"[SCARB1, LCP1]"
2,GO_Biological_Process_2023,Actin Filament Bundle Assembly (GO:0051017),0.000255,0.052691,228.649291,3,"[BAIAP2L1, LCP1, AIF1L]"
3,GO_Biological_Process_2023,Actin Filament Bundle Organization (GO:0061572),0.000255,0.052691,228.649291,3,"[BAIAP2L1, LCP1, AIF1L]"
4,MSigDB_Hallmark_2020,Estrogen Response Early,0.006786,0.067856,28.314458,4,"[SCARB1, ELF1, STC2, KRT8]"
5,MSigDB_Hallmark_2020,Glycolysis,0.006786,0.067856,28.314458,4,"[FUT8, GLCE, STC2, LHPP]"
6,MSigDB_Hallmark_2020,Hypoxia,0.006786,0.067856,28.314458,4,"[SELENBP1, SCARB1, STC2, SLC2A3]"
7,GO_Biological_Process_2023,Regulation Of Cell-Substrate Adhesion (GO:0010810),0.000601,0.084402,149.889349,3,"[AGR2, RAC2, ARL2]"
8,GO_Biological_Process_2023,Positive Regulation Of Lipid Localization (GO:1905954),0.000612,0.084402,504.623799,2,"[LRP1, LCP1]"
9,GO_Biological_Process_2023,Actin Crosslink Formation (GO:0051764),0.000747,0.088206,436.513208,2,"[BAIAP2L1, LCP1]"


In [10]:
for d in DIRECTIONS:
    print(f"\n========== R239G {d} ==========")
    display(top_terms("R239G", d))


========== R239G up ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,GO_Biological_Process_2023,Axonogenesis (GO:0007409),0.000011,0.007259,154.611294,6,"[EPHA5, EPHA7, SLITRK4, WNT5A, MAP1A, B4GALT5]"
1,GO_Biological_Process_2023,Regulation Of Cellular Component Organization (GO:0051128),0.000057,0.018414,133.307480,5,"[SOCS2, SLITRK4, WNT5A, DAB2IP, GFAP]"
2,Reactome_Pathways_2024,The Activation of Arylsulfatases,0.000548,0.054574,523.470491,2,"[ARSL, ARSD]"
3,Reactome_Pathways_2024,Glycosphingolipid Metabolism,0.000517,0.054574,160.995530,3,"[ARSL, ARSD, B4GALT5]"
4,Reactome_Pathways_2024,Lysine Catabolism,0.000464,0.054574,588.522912,2,"[AADAT, DHTKD1]"
5,GO_Biological_Process_2023,Regulation Of Protein Catabolic Process (GO:0042176),0.000329,0.070463,107.792291,4,"[WNT5A, ODC1, DAB2IP, GFAP]"
6,GO_Biological_Process_2023,Secondary Palate Development (GO:0062009),0.000548,0.087884,523.470491,2,"[SMAD2, WNT5A]"
7,GO_Biological_Process_2023,Regulation Of Epithelial To Mesenchymal Transition (GO:0010717),0.001468,0.128054,95.301161,3,"[SMAD2, DAB2IP, VASN]"
8,GO_Biological_Process_2023,Positive Regulation Of Cell Projection Organization (GO:0031346),0.003989,0.128054,56.035629,3,"[WNT5A, DAB2IP, SERPINI1]"
9,GO_Biological_Process_2023,Positive Regulation Of Neuron Death (GO:1901216),0.003798,0.128054,133.397678,2,"[EPHA7, WNT5A]"



========== R239G down ==========


,library,term,pvalue,adjusted_pvalue,combinedscore,n_overlap,overlapping_genes
0,MSigDB_Hallmark_2020,Estrogen Response Late,0.000271,0.008660,60.354100,6,"[SOX3, ALDH3B1, AGR2, SFN, CXCL14, AFF1]"
1,Reactome_Pathways_2024,Post-translational Protein Phosphorylation,0.000116,0.023236,104.765552,5,"[ITIH2, IGFBP5, AHSG, APOA1, AFP]"
2,Reactome_Pathways_2024,Platelet Degranulation,0.000268,0.023236,78.755927,5,"[POTEKP, AHSG, SERPINF2, APOA1, A2M]"
3,Reactome_Pathways_2024,Neutrophil Degranulation,0.000276,0.023236,38.218516,9,"[BST2, ARSA, CSTB, AHSG, GAA, ALDH3B1, GGH, SLC2A3, DNASE1L1]"
4,Reactome_Pathways_2024,Response to Elevated Platelet Cytosolic Ca2+,0.000320,0.023236,74.030163,5,"[POTEKP, AHSG, SERPINF2, APOA1, A2M]"
5,Reactome_Pathways_2024,Regulation of IGF Transport and Uptake by Insulin-like Growth Fact...,0.000231,0.023236,82.878945,5,"[ITIH2, IGFBP5, AHSG, APOA1, AFP]"
6,MSigDB_Hallmark_2020,Estrogen Response Early,0.002001,0.032019,37.397878,5,"[SOX3, ELF1, ALDH3B1, SFN, AFF1]"
7,Reactome_Pathways_2024,HDL Assembly,0.000539,0.032597,573.979943,2,"[APOA1, A2M]"
8,GO_Biological_Process_2023,Positive Regulation Of Hydrolase Activity (GO:0051345),0.000135,0.037610,74.771336,6,"[USP6NL, WRN, ALS2, CAV2, DOCK8, APOA1]"
9,GO_Biological_Process_2023,Cholesterol Homeostasis (GO:0042632),0.000299,0.037610,111.428365,4,"[IL18, APOA1, LCP1, HDAC9]"
